# train_final.ipynb — MetricHunter Final Model Eğitimi

Bu notebook'u bir kez çalıştırın. `models/` altında 7 dosya/klasör üretilir:
- `commit_rf.joblib` — Commit tahmini RF modeli
- `scaler_commit.joblib` — Commit feature scaler
- `bug_rf_base.joblib` — Bug stacking base RF
- `bug_ag_base/` — Bug stacking base AutoGluon
- `bug_meta_lr.joblib` — Bug stacking meta LR
- `scaler_bug.joblib` — Bug feature scaler
- `feature_names.json` — Feature listeleri

In [1]:
import os
import json
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from autogluon.tabular import TabularPredictor

RANDOM_STATE = 42
DATA_PATH = Path("output/dataset_model_filtered_20260331_1613.csv")
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

print("Imports OK")

Imports OK


In [2]:
# --- Feature Tanımları ---
STATIC_METRIC_COLS = [
    "loc", "lloc", "sloc", "comments", "multi", "blank", "single_comments",
    "cc_mean", "cc_max", "cc_total", "num_functions",
    "h_vocabulary", "h_length", "h_volume", "h_difficulty",
    "h_effort", "h_bugs", "h_time", "h_calculated_length",
    "maintainability_index", "comment_ratio", "doc_ratio"
]

DERIVED_METRIC_COLS = [
    "complexity_density", "comment_per_function", "avg_function_length", "effort_per_line"
]

PROCESS_METRIC_COLS = [
    "commit_count", "bug_count", "n_authors", "file_age_days",
    "churn_total", "avg_churn_per_commit", "max_single_churn", "recent_commits_90d"
]

PROJECT_COLS = ["stars", "contributor_count", "project_age_days"]

FEATURES_ALL_LABEL = STATIC_METRIC_COLS + DERIVED_METRIC_COLS + PROJECT_COLS  # 29
FEATURES_ALL_BUG = [c for c in STATIC_METRIC_COLS + DERIVED_METRIC_COLS + PROCESS_METRIC_COLS + PROJECT_COLS
                    if c != "bug_count"]  # 36

print(f"Commit features: {len(FEATURES_ALL_LABEL)}")
print(f"Bug features: {len(FEATURES_ALL_BUG)}")
print("FEATURES_ALL_LABEL:", FEATURES_ALL_LABEL)
print("FEATURES_ALL_BUG:", FEATURES_ALL_BUG)

Commit features: 29
Bug features: 36
FEATURES_ALL_LABEL: ['loc', 'lloc', 'sloc', 'comments', 'multi', 'blank', 'single_comments', 'cc_mean', 'cc_max', 'cc_total', 'num_functions', 'h_vocabulary', 'h_length', 'h_volume', 'h_difficulty', 'h_effort', 'h_bugs', 'h_time', 'h_calculated_length', 'maintainability_index', 'comment_ratio', 'doc_ratio', 'complexity_density', 'comment_per_function', 'avg_function_length', 'effort_per_line', 'stars', 'contributor_count', 'project_age_days']
FEATURES_ALL_BUG: ['loc', 'lloc', 'sloc', 'comments', 'multi', 'blank', 'single_comments', 'cc_mean', 'cc_max', 'cc_total', 'num_functions', 'h_vocabulary', 'h_length', 'h_volume', 'h_difficulty', 'h_effort', 'h_bugs', 'h_time', 'h_calculated_length', 'maintainability_index', 'comment_ratio', 'doc_ratio', 'complexity_density', 'comment_per_function', 'avg_function_length', 'effort_per_line', 'commit_count', 'n_authors', 'file_age_days', 'churn_total', 'avg_churn_per_commit', 'max_single_churn', 'recent_commits_

In [3]:
# --- Veri Yükleme ve Temizlik ---
df = pd.read_csv(DATA_PATH)
print(f"Ham veri: {len(df)} satır, {df['project'].nunique()} proje")

# 1. MI = 0 → radon hesaplayamadı, bu satırları çıkar
mi_zero = (df["maintainability_index"] == 0).sum()
df = df[df["maintainability_index"] > 0]
print(f"MI=0 elendi: {mi_zero} dosya")

# 2. NaN temizle
all_features = list(set(FEATURES_ALL_LABEL + FEATURES_ALL_BUG))
before = len(df)
df = df.dropna(subset=all_features + ["label", "has_bug"])
print(f"NaN elendi: {before - len(df)} dosya")

print(f"\nTemiz veri: {len(df)} satır, {df['project'].nunique()} proje")
print(f"Label dağılımı: {df['label'].value_counts().to_dict()}")
print(f"Has_bug dağılımı: {df['has_bug'].value_counts().to_dict()}")

Ham veri: 10535 satır, 94 proje
MI=0 elendi: 0 dosya
NaN elendi: 0 dosya

Temiz veri: 10535 satır, 94 proje
Label dağılımı: {0: 5695, 1: 4840}
Has_bug dağılımı: {1: 5286, 0: 5249}


In [4]:
# --- Sensitivity Filtering ---
# Commit (label): min=25, max=80
# Bug (has_bug): min=25, max=100

def sensitivity_filter(df, min_files, max_files, random_state=42):
    project_sizes = df.groupby("project").size()
    keep = project_sizes[project_sizes >= min_files].index
    filtered = df[df["project"].isin(keep)].copy()
    if max_files is not None:
        filtered = filtered.groupby("project", group_keys=False).apply(
            lambda x: x.sample(min(len(x), max_files), random_state=random_state)
        ).reset_index(drop=True)
    return filtered

df_label = sensitivity_filter(df, min_files=25, max_files=80)
df_bug   = sensitivity_filter(df, min_files=25, max_files=100)

print(f"Commit verisi: {len(df_label)} dosya, {df_label['project'].nunique()} proje")
print(f"Bug verisi   : {len(df_bug)} dosya, {df_bug['project'].nunique()} proje")
print(f"Label dağılımı: {df_label['label'].value_counts().to_dict()}")
print(f"Has_bug dağılımı: {df_bug['has_bug'].value_counts().to_dict()}")

Commit verisi: 5742 dosya, 81 proje
Bug verisi   : 6773 dosya, 81 proje
Label dağılımı: {0: 3075, 1: 2667}
Has_bug dağılımı: {1: 3437, 0: 3336}


C:\Users\bm123\AppData\Local\Temp\ipykernel_40324\2578874525.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  filtered = filtered.groupby("project", group_keys=False).apply(
C:\Users\bm123\AppData\Local\Temp\ipykernel_40324\2578874525.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  filtered = filtered.groupby("project", group_keys=False).apply(


In [5]:
# --- Commit Modeli (Random Forest) ---
print("=" * 50)
print("COMMIT MODELİ EĞİTİLİYOR...")
print("=" * 50)

X_label = df_label[FEATURES_ALL_LABEL].values
y_label = df_label["label"].values

scaler_commit = StandardScaler()
X_label_s = scaler_commit.fit_transform(X_label)

commit_rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
commit_rf.fit(X_label_s, y_label)

joblib.dump(commit_rf, MODELS_DIR / "commit_rf.joblib")
joblib.dump(scaler_commit, MODELS_DIR / "scaler_commit.joblib")
print("Kaydedildi: commit_rf.joblib, scaler_commit.joblib")

COMMIT MODELİ EĞİTİLİYOR...
Kaydedildi: commit_rf.joblib, scaler_commit.joblib


In [6]:
# --- Bug Modeli (Stacking: RF + AutoGluon → LR) ---
print("=" * 50)
print("BUG MODELİ EĞİTİLİYOR (STACKING)...")
print("=" * 50)

from sklearn.model_selection import train_test_split

X_bug = df_bug[FEATURES_ALL_BUG].values
y_bug = df_bug["has_bug"].values

scaler_bug = StandardScaler()
X_bug_s = scaler_bug.fit_transform(X_bug)

# 80/20 split: base_train / meta_train (shuffled + stratified)
X_base, X_meta, y_base, y_meta = train_test_split(
    X_bug_s, y_bug, test_size=0.20, random_state=RANDOM_STATE, stratify=y_bug
)

print(f"Base train: {len(X_base)}, Meta train: {len(X_meta)}")
print(f"Base label dağılımı: 0={sum(y_base==0)}, 1={sum(y_base==1)}")
print(f"Meta label dağılımı: 0={sum(y_meta==0)}, 1={sum(y_meta==1)}")

# Base Model 1: Random Forest
bug_rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
bug_rf.fit(X_base, y_base)
rf_meta_proba = bug_rf.predict_proba(X_meta)[:, 1]
print(f"RF meta proba shape: {rf_meta_proba.shape}")

joblib.dump(bug_rf, MODELS_DIR / "bug_rf_base.joblib")
print("Kaydedildi: bug_rf_base.joblib")

BUG MODELİ EĞİTİLİYOR (STACKING)...
Base train: 5418, Meta train: 1355
Base label dağılımı: 0=2669, 1=2749
Meta label dağılımı: 0=667, 1=688
RF meta proba shape: (1355,)
Kaydedildi: bug_rf_base.joblib


In [7]:
# Base Model 2: AutoGluon
ag_train_df = pd.DataFrame(X_base, columns=FEATURES_ALL_BUG)
ag_train_df["has_bug"] = y_base

ag_predictor = TabularPredictor(
    label="has_bug",
    eval_metric="f1_weighted",
    path=str(MODELS_DIR / "bug_ag_base"),
    verbosity=1
).fit(
    ag_train_df,
    time_limit=300,
    presets="good_quality",
    dynamic_stacking=False
)

ag_meta_input = pd.DataFrame(X_meta, columns=FEATURES_ALL_BUG)
ag_meta_pred = ag_predictor.predict_proba(ag_meta_input)
if isinstance(ag_meta_pred, pd.DataFrame):
    ag_meta_proba = ag_meta_pred[1].values
else:
    ag_meta_proba = ag_meta_pred[:, 1]

print(f"AutoGluon meta proba shape: {ag_meta_proba.shape}")
print("Kaydedildi: bug_ag_base/")

AutoGluon infers your prediction problem is: 'binary' (because only two unique label-values observed).
	If 'binary' is not the correct problem_type, please manually specify the problem_type parameter during Predictor init (You may specify problem_type as one of: ['binary', 'multiclass', 'regression', 'quantile'])
C:\Users\bm123\AppData\Local\Programs\Python\Python310\lib\site-packages\ray\_private\worker.py:2062: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
C:\Users\bm123\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\compose\_column_transformer.py:975: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(


AutoGluon meta proba shape: (1355,)
Kaydedildi: bug_ag_base/


In [8]:
# Meta Model: Logistic Regression
meta_features = np.column_stack([rf_meta_proba, ag_meta_proba])

bug_meta_lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
bug_meta_lr.fit(meta_features, y_meta)

joblib.dump(bug_meta_lr, MODELS_DIR / "bug_meta_lr.joblib")
joblib.dump(scaler_bug, MODELS_DIR / "scaler_bug.joblib")
print("Kaydedildi: bug_meta_lr.joblib, scaler_bug.joblib")

Kaydedildi: bug_meta_lr.joblib, scaler_bug.joblib


In [9]:
# --- Feature İsimlerini Kaydet ---
feature_names = {
    "commit": FEATURES_ALL_LABEL,
    "bug": FEATURES_ALL_BUG
}
with open(MODELS_DIR / "feature_names.json", "w") as f:
    json.dump(feature_names, f, indent=2)
print("Kaydedildi: feature_names.json")

Kaydedildi: feature_names.json


In [ ]:
# --- Doğrulama: Modelleri yükleyip test tahmini yap ---
print("\n" + "=" * 50)
print("DOĞRULAMA")
print("=" * 50)

# Modelleri yükle
_cr = joblib.load(MODELS_DIR / "commit_rf.joblib")
_sc = joblib.load(MODELS_DIR / "scaler_commit.joblib")
_br = joblib.load(MODELS_DIR / "bug_rf_base.joblib")
_bm = joblib.load(MODELS_DIR / "bug_meta_lr.joblib")
_sb = joblib.load(MODELS_DIR / "scaler_bug.joblib")

# require_py_version_match=False: Flask farklı Python sürümüyle çalışıyor olsa bile yükler
_ag = TabularPredictor.load(str(MODELS_DIR / "bug_ag_base"), require_py_version_match=False)

with open(MODELS_DIR / "feature_names.json") as f:
    _fn = json.load(f)

# Test tahmini (ilk satır)
test_row_commit = df_label[FEATURES_ALL_LABEL].iloc[:1]
test_scaled_commit = _sc.transform(test_row_commit)
pred_commit = _cr.predict(test_scaled_commit)[0]
prob_commit = _cr.predict_proba(test_scaled_commit)[0][1]
print(f"Commit test: pred={pred_commit}, prob={prob_commit:.3f}")

test_row_bug = df_bug[FEATURES_ALL_BUG].iloc[:1]
test_scaled_bug = _sb.transform(test_row_bug)
rf_p = _br.predict_proba(test_scaled_bug)[:, 1]
ag_inp = pd.DataFrame(test_scaled_bug, columns=FEATURES_ALL_BUG)
ag_p = _ag.predict_proba(ag_inp)
if isinstance(ag_p, pd.DataFrame):
    ag_p = ag_p[1].values
else:
    ag_p = ag_p[:, 1]
meta_inp = np.column_stack([rf_p, ag_p])
pred_bug = _bm.predict(meta_inp)[0]
prob_bug = _bm.predict_proba(meta_inp)[0][1]
print(f"Bug test: pred={pred_bug}, prob={prob_bug:.3f}")

print("\nTum modeller basariyla egitildi ve dogrulandı!")
print("\nmodels/ klasoru icerigi:")
for f in sorted(MODELS_DIR.iterdir()):
    print(f"  {f.name}")
